# `embedder` (EmbedderModule)
- **Category**: Logic (Embedding)
- **Role**: 분해된 서브쿼리 리스트(`SubqueriesDTO`)를 3072차원 고차원 밀집 벡터 딕셔너리로 일괄 인코딩합니다.


In [ ]:
import sys
from pathlib import Path
import json

# 프로젝트 루트 경로 등록
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().name == "modules" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def print_io(title: str, input_data: dict, output_data: dict):
    print("=" * 70)
    print(f"📌 [Module Execution] {title}")
    print("=" * 70)
    print("\n📥 [Input DTO]")
    print(json.dumps(input_data, indent=2, ensure_ascii=False))
    print("\n📤 [Output Result]")
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
    print("\n")


In [ ]:
from unittest.mock import MagicMock
from modules.embedding.query_embedder import EmbedderModule, EmbedderInputDTO, EmbedderConfigDTO

mock_encoder = MagicMock()
mock_encoder.model_name = "text-embedding-3-large"
mock_encoder.dimension = 3072
mock_encoder.encode.return_value = [
    [0.0123, -0.0456, 0.0789] + [0.0] * 3069,
    [-0.0321, 0.0654, -0.0987] + [0.0] * 3069
]

module = EmbedderModule(encoder=mock_encoder)

sample_input = {
    "query_input": {
        "query_context": {
            "question_id": "QUERY-001",
            "question_text": "2023년 삼성전자 영업이익과 2022년 대비 증감율"
        },
        "items": [
            {
                "company": "삼성전자",
                "sheet": "손익계산서",
                "row_header": "영업이익",
                "column_header": "2022",
                "cell_value": "?"
            },
            {
                "company": "삼성전자",
                "sheet": "손익계산서",
                "row_header": "영업이익",
                "column_header": "2023",
                "cell_value": "?"
            }
        ]
    }
}
input_dto = EmbedderInputDTO(**sample_input)
output = module.run(input_dto, config=EmbedderConfigDTO(model="text-embedding-3-large", dimension=3072))

summary_output = {
    "query_context": output["query_context"],
    "items": {
        q: {"dimension": len(vec), "preview": vec[:3]}
        for q, vec in output["items"].items()
    }
}
print_io("embedder (EmbedderModule)", sample_input, summary_output)
